# CustomCPU graph atlas

Generated from read-only Logisim source files. The port/net graph is canonical; visual trees are derived views.

In [ ]:
from pathlib import Path
import json
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'armv4t.circ').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'armv4t.circ').exists(), 'Run from inside the CustomCPU repository'
ATLAS = ROOT / 'ipynb'


## Reading the atlas

- `*.json` contains every parsed component, port, raw wire segment, net, and directed connection.
- `*.port_net.graphml` is the lossless bipartite electrical view.
- `*.signal.graphml` directs every modeled driver toward its sinks.
- `*.component.graphml` collapses ports for readable block diagrams.
- `*.condensation.graphml` collapses feedback strongly-connected components into a DAG.
- `data/diff/` compares `armv4t.circ` with `debug_armv4t.circ`.

Human instructions use `source.output.wire -> destination.input_pin`; coordinates are machine identifiers only.

In [ ]:
arm = json.loads((ATLAS/'data/armv4t/manifest.json').read_text())
debug = json.loads((ATLAS/'data/debug/manifest.json').read_text())
delta = json.loads((ATLAS/'data/diff/manifest.json').read_text())
pd.DataFrame([{
 'file': arm['source'], 'sha256': arm['sha256'], 'circuits': arm['circuit_count']},
 {'file': debug['source'], 'sha256': debug['sha256'], 'circuits': debug['circuit_count']}])

## Complete circuit inventory

In [ ]:
a = pd.DataFrame(arm['circuits']).set_index('name').add_prefix('arm_')
b = pd.DataFrame(debug['circuits']).set_index('name').add_prefix('debug_')
a.join(b, how='outer')

## Hierarchy graph

In [ ]:
h = nx.read_graphml(ATLAS/'data/armv4t/hierarchy.graphml')
plt.figure(figsize=(18, 14))
pos = nx.spring_layout(h, seed=7, k=1.2)
nx.draw_networkx(h, pos, node_size=1400, font_size=7, arrows=True)
plt.axis('off'); plt.show()

## File-to-file changes

In [ ]:
pd.DataFrame(delta['circuits']).query('changed == True')